# arms · Stats — the per-arm inferential tables  `[EVAL]`

The **full statistical tables** behind the per-arm figures in `arms/outcomes`, `arms/questionnaires`
and `arms/heterogeneity` — kept here so those notebooks stay figure-led. Every table is
full-conversation eval, **all four arms on one axis** (`PTO_LA0`, `PTO_LA5`, `GRPO_LA0`,
`GRPO_LA5`), each arm measured against **its own base** and paired by the 96 shared personas
(`persona_id`, never `file_index` — the personas are reshuffled every iteration). Thin arms
(< 3 scored iterations) are dropped to avoid NaN rows.

**What this family answers.** *Does each arm move, and how much?* — the within-arm questions
(target vs base, iteration as a within-persona factor, the per-iteration vs-base sweep, climb rate,
whether the rubrics collapse to one factor). It does **not** answer any *between-arm* question:
the method contrast (PTO vs GRPO) lives in `method/contrast`, the look-ahead contrast (K=0 vs K=5)
in `lookahead/reward`, and the cost axis in `compute/cost` — read those before comparing rows of
different arms here.

**Grader.** Rendered once per judge: `<judge>/` = the grader whose scores the tables carry
(`gpt-4o-mini` = the primary oracle, i.e. the training reward; `claude-haiku-4-5` = the held-out
judge). ⚠ Never average the two leaves — the primary WAS the reward, the held-out judge was not.

**Support.** Each arm's *final* rows sit at its own last scored iteration, and how far an arm is
scored can be grader-dependent; both are derived per render - see the support line every caption
carries, which is empty when every arm reaches the same iteration. *(Corrected 2026-08-25: this
said `GRPO_LA5` "stops short of the other three arms" and that its Friedman / slope statistics
"span fewer states". That arm finished at iteration 10; all four arms now span 11 states.)*

Exports → `results/arms/stats/tables/<judge>/`.

## 0 · Confirmatory vs exploratory — read this first  `[EVAL]`

**Multiplicity scope.** Every *p*-value below is Holm-corrected **within** its own family (across
rubrics within one arm × target, or across iterations within one arm-vs-base sweep), but
corrections are **not** pooled across the many families of this EDA. To keep the inference honest,
the analyses split into a small pre-registered **confirmatory** set (the thesis claims) and a
larger **exploratory** set (hypothesis-generating — report descriptively; don't over-read an
isolated star).

**Confirmatory (primary hypotheses) that this family carries.**
- **Each arm improves over its own base** on the primary metric (Q1+Q2), at both the *final* and
  the *best* iteration (§1, `target` column of `main_results`).
- **The reward-hacking signature on the metrics the reward never saw** — MICI (MI-inconsistency,
  lower = better) rises alongside the global-eval scores; the per-arm numbers are the MICI rows of
  §1/§3, the synthesis is in `arms/validity`.

**Confirmatory claims that live elsewhere.** PTO > GRPO on Q1+Q2 (matched final and best-vs-best)
→ `method/contrast`; the look-ahead lever K=0 vs K=5 → `lookahead/reward` (matched iteration) and
`compute/cost` (matched budget — the two axes can disagree in sign, so quote both or say which).

**Exploratory (hypothesis-generating).** Per-item / subscale trajectories (`arms/questionnaires`);
per-persona-trait subgroups (`arms/heterogeneity`); the GRPO iter-9 anomaly check
(`arms/validity`); the full per-iteration vs-base sweeps here (§3); the factor structure (§4).

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting, stats
from eda_analysis.constants import judge_dirname
cfg = eda_analysis.EdaConfig(family="arms/stats", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the banner reset_results just removed (it lives under figures/<judge>/)

GRADER = judge_dirname(S.JUDGE) + (" (primary oracle = the training reward)" if not S.JUDGE else " (held-out judge)")
ARMS_ = sorted(S.SCORES.arm.unique())
THIN = stats.thin_arms(S.SCORES)
# Derived, per grader: an arm's final/last rows sit at its own last scored iteration, and how far the
# score lake covers it differs by judge - so this sentence is read off S.SCORES, never written down.
CENSOR = eda_analysis.support_note(S.SCORES, subject=f"no later state scored by {judge_dirname(S.JUDGE)}")
PAIR = "persona-paired (persona_id, N=96 shared personas; never file_index)"
print("grader:", GRADER, "| arms:", ARMS_, "| thin (dropped):", THIN)

## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × metric): the FINAL and the BEST iteration vs base in one table (column
`target`) — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm across rubrics within arm × target),
bootstrap 95 % CI, trajectory ρ / slope. *Best* = each arm's peak iteration on its **own training
oracle** (`best_per_experiment`), so the best-row is a model-selection statement, not a
post-hoc pick on the eval metric. **Sign:** `delta = target − base`, so + ⇒ the trained model
scores higher; on `MICI` (lower = better) a positive Δ is a *worsening*.

In [ ]:
MR  = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"),  S.SCORES)
MRall = pd.concat([MR.assign(target="final"), MRb.assign(target="best")], ignore_index=True)
MRall = MRall[["arm", "rubric", "target", "base", "target_iter"] +
              [c for c in MRall.columns if c not in ("arm", "rubric", "target", "base", "target_iter")]]
display(MRall)
exports.save_table(MRall, "main_results", caption=(
    f"FINAL and BEST iteration vs base per arm x metric (column `target`), all four arms; grader = {GRADER}. "
    f"{PAIR}: delta = target - base (+ => trained model higher; MICI is lower-better so + is worse there), "
    "paired Cohen's dz + label, Wilcoxon p (Holm across rubrics within arm x target), bootstrap 95% CI "
    "(BOOT_SEED), trajectory Spearman rho / OLS slope. BEST = the arm's peak iteration on its own training "
    f"oracle (best_per_experiment). {CENSOR} Thin arms (<3 scored iters) dropped."))

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm ×
rubric — the matched-persona-correct omnibus (a persona × iteration pivot, complete after persona
recovery), preferred over an independent-group Kruskal–Wallis for this design. `k_iters` is the
number of scored iterations the pivot spans (base included) - the record of each arm's support,
currently 11 for all four arms under both graders.

In [ ]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in ARMS_ for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
exports.save_table(FR.round(4), "friedman_omnibus", caption=(
    f"Friedman repeated-measures omnibus across iterations per arm x rubric (chi2, p, Kendall's W), all four arms; "
    f"grader = {GRADER}. Unit = persona (persona_id x iteration pivot, base included; n_personas = complete rows). "
    f"No sign convention (an omnibus). {CENSOR} Thin arms dropped."))

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN
base, one merged table with an `arm` column). Holm is applied across the iterations *within* each
arm's sweep. **Sign:** `mean_delta = iteration − base` (+ ⇒ the later policy scores higher).

In [ ]:
frames = []
for arm in ARMS_:
    if arm in THIN: continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if not PV.empty: frames.append(PV)
if frames:
    VB = pd.concat(frames, ignore_index=True)[["arm", "iteration", "n", "mean_delta", "dz", "p", "p_holm", "ci_low", "ci_high"]].round(4)
    display(VB)
    exports.save_table(VB, "vs_base_paired", caption=(
        f"Each arm x iteration vs its OWN base on Q1+Q2 -- one merged table (column `arm`), all four arms; grader = {GRADER}. "
        f"{PAIR}: mean_delta = iteration - base (+ => later policy higher), Wilcoxon p, dz, Holm p (across iterations "
        f"WITHIN each arm), bootstrap 95% CI (BOOT_SEED). {CENSOR}"))
else:
    print("no non-thin arms scored.")

## 4 · Climb rate and factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm × metric (climb rate; `peak_iter` vs
`final_iter` flags a post-peak regression); and the rubric PCA (PC1 share → do the rubrics collapse
to ~one latent factor?). **Note:** `slope_by_arm` reports ρ / slope but no *p* — the pooled
persona × iteration rows are not independent, so trajectory *p*-values are descriptive; the
repeated-measures-correct omnibus is the Friedman table in §2. **PCA caveats:** the PC1 share is
computed on the canonical 10-metric factor space (`WARMTH_RUBRICS + EXTRA_METRICS`), **pooled**
over all conversations and iterations of the arm; a lower share when less-correlated metrics are
appended is *partly mechanical*, so read it as "a second dimension exists", not a calibrated
effect size. **Bootstrap CIs** everywhere (§1, §3) use the package seed (`BOOT_SEED`) →
reproducible but not a source of independent uncertainty beyond the resample.

In [ ]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, m)
      for a in ARMS_ for m in S.METRICS]), S.SCORES)
SLv = SL[["arm", "metric", "spearman_rho", "ols_slope", "peak_iter", "final_iter"]].round(4)
display(SLv)
exports.save_table(SLv, "slope_by_arm", caption=(
    f"Per-iteration OLS slope + Spearman rho per arm x metric (climb rate; peak_iter vs final_iter flags a "
    f"post-peak regression), all four arms; grader = {GRADER}. Pooled persona x iteration rows (unit = conversation), "
    "so NO p is reported -- descriptive; the repeated-measures omnibus is friedman_omnibus. + slope => the metric "
    f"climbs with iteration (MICI is lower-better, so + is worse there). {CENSOR} Thin arms dropped."))

PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in ARMS_ if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
exports.save_table(PCA, "rubric_pca_pc1", caption=(
    f"Variance explained by PC1 of the standardized rubric scores per arm (canonical 10-metric factor space, pooled "
    f"over every conversation x iteration of the arm), all four arms; grader = {GRADER}. A dominant PC1 => the rubrics "
    "~ one latent factor, so 'every metric up' is weak evidence of multi-skill gain. Unit = conversation; no pairing, "
    f"no sign. {CENSOR}"))

## 5 · Artifact index
Drop captions whose artifact no longer exists, then refresh `results/arms/INDEX.md` (+ the root
`results/INDEX.md`) so the family map stays complete whatever notebook rendered last.

In [ ]:
exports.prune_orphan_captions(); exports.build_index()